# 04 — SQL Analytics: WO-1 Readiness Audit

## Objective

This notebook records the WO-1 diagnostic workflow for AN0, AN1, and the gated first part of AN2. The SQL files under `sql/04_analytics/` are the source of truth; this notebook calls them by stable query ID and does not duplicate their SQL text.

## Execution and safety rules

- Execution environment: the repository root and the project Python environment.
- The helper opens a new PostgreSQL connection for each query with read-only session settings, a 120-second statement timeout, and a 5-second lock timeout.
- Outputs are capped at 500 rows. Current WO-1 queries return summary tables; the full `student_vle` table is never loaded into Pandas.
- Scope exclusion: RAW loading, STAGING creation, quality-registry seeding, and all DDL.
- If AN0.01 fails its safety check, stop. If AN1.01 does not PASS for both tables, do not run AN2.01 or AN2.02.
- Each query result must be reviewed and interpreted before the work order is marked complete.

## Work-order status

**Execution status:** Pending local execution. No live database results are recorded in this notebook yet.

## 0. Load the read-only query helper

The helper resolves SQL files by ID and returns a capped result table with execution time and file path. It reads local credentials from `.env` without printing them.

In [ ]:
# 1. Load the project root and canonical query runner.
from pathlib import Path
import sys
from IPython.display import display

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "sql" / "04_analytics").exists():
    PROJECT_ROOT = Path.cwd().parent
if not (PROJECT_ROOT / "sql" / "04_analytics").exists():
    raise FileNotFoundError("Open this notebook from the repository or notebooks folder.")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.analytics.db import run_query

query_results = {}
print("Project root:", PROJECT_ROOT.name)
print("Canonical SQL folder found:", (PROJECT_ROOT / "sql" / "04_analytics").exists())

**Validation criteria:** The canonical SQL directory resolves correctly, and connection secrets are never displayed. An import failure must be resolved before query execution; local credential files remain private.

## AN0.01 — Database identity and session safety

**Question / Why:** Confirm the database and user before any analysis. Stop if the active database is not `oulad_analytics` or the session is not read-only.

**Expectation:** The intended database is active and read-only settings are enabled.

**Read as:** This query returns metadata only and should finish quickly.

In [ ]:
# 1. Verify the active database and read-only session before continuing.
result = run_query("AN0.01")
query_results["AN0.01"] = result
display(result["data"])
print(f"Query file: {result['query_file']}")
print(f"Elapsed seconds: {result['elapsed_seconds']}")
print(f"Displayed rows: {result['rows_displayed']}; truncated: {result['display_truncated']}")

row = result["data"].iloc[0]
if row["database_name"] != "oulad_analytics" or str(row["transaction_read_only"]).lower() != "on":
    raise RuntimeError("Safety gate failed. Stop the notebook and investigate the active connection.")

**After execution — interpretation and decision (complete after reviewing the output):**

- **Observed:** Record the actual values and any difference from expectation.
- **Alternative explanations:** If results differ, state at least two plausible explanations.
- **Discriminating evidence:** Identify the next read-only check that can separate those explanations.
- **Decision:** State whether the next stage is approved, paused, or needs investigation.
- **Limitations:** Note the grain, denominator, join path, and whether the result is descriptive or associative.
- **📸 Screenshot:** Capture only if the result provides a clear, verified technical story worth sharing later; otherwise no screenshot is needed.

## AN0.02 — Existing quality registry snapshot

**Question / Why:** Read the current registry as evidence. Completed engineering checks and seed scripts are outside this work order.

**Expectation:** Historical reference: 29 rows/codes, 25 PASS, 4 REVIEW, 0 FAIL. Historical execution timestamps may be NULL.

**Read as:** Differences are a reconciliation question, not a reason to rewrite the registry.

In [ ]:
# 1. Execute the canonical SQL file and display only its bounded result set.
result = run_query("AN0.02")
query_results["AN0.02"] = result
display(result["data"])
print(f"Query file: {result['query_file']}")
print(f"Elapsed seconds: {result['elapsed_seconds']}")
print(f"Displayed rows: {result['rows_displayed']}; truncated: {result['display_truncated']}")

**After execution — interpretation and decision (complete after reviewing the output):**

- **Observed:** Record the actual values and any difference from expectation.
- **Alternative explanations:** If results differ, state at least two plausible explanations.
- **Discriminating evidence:** Identify the next read-only check that can separate those explanations.
- **Decision:** State whether the next stage is approved, paused, or needs investigation.
- **Limitations:** Note the grain, denominator, join path, and whether the result is descriptive or associative.
- **📸 Screenshot:** Capture only if the result provides a clear, verified technical story worth sharing later; otherwise no screenshot is needed.

## AN0.03 — STAGING row-count reconciliation

**Question / Why:** Confirm whether the live database still matches the documented counts. PostgreSQL computes exact counts; the notebook receives only seven summary rows.

**Expectation:** 22, 206, 32,593, 32,593, 173,912, 6,364, and 10,655,280 rows in documented table order.

**Read as:** This query scans `student_vle` and is subject to the configured 120-second timeout. STAGING creation is outside this work order.

In [ ]:
# 1. Execute the canonical SQL file and display only its bounded result set.
result = run_query("AN0.03")
query_results["AN0.03"] = result
display(result["data"])
print(f"Query file: {result['query_file']}")
print(f"Elapsed seconds: {result['elapsed_seconds']}")
print(f"Displayed rows: {result['rows_displayed']}; truncated: {result['display_truncated']}")

**After execution — interpretation and decision (complete after reviewing the output):**

- **Observed:** Record the actual values and any difference from expectation.
- **Alternative explanations:** If results differ, state at least two plausible explanations.
- **Discriminating evidence:** Identify the next read-only check that can separate those explanations.
- **Decision:** State whether the next stage is approved, paused, or needs investigation.
- **Limitations:** Note the grain, denominator, join path, and whether the result is descriptive or associative.
- **📸 Screenshot:** Capture only if the result provides a clear, verified technical story worth sharing later; otherwise no screenshot is needed.

## AN0.04 — STAGING sizes and indexes

**Question / Why:** Inspect the actual catalog state before making any performance decision.

**Expectation:** Index presence is established from the live catalog; CTAS does not guarantee that source indexes or constraints were copied.

**Read as:** No indexes are created. Record sizes and existing index definitions for later performance planning.

In [ ]:
# 1. Execute the canonical SQL file and display only its bounded result set.
result = run_query("AN0.04")
query_results["AN0.04"] = result
display(result["data"])
print(f"Query file: {result['query_file']}")
print(f"Elapsed seconds: {result['elapsed_seconds']}")
print(f"Displayed rows: {result['rows_displayed']}; truncated: {result['display_truncated']}")

**After execution — interpretation and decision (complete after reviewing the output):**

- **Observed:** Record the actual values and any difference from expectation.
- **Alternative explanations:** If results differ, state at least two plausible explanations.
- **Discriminating evidence:** Identify the next read-only check that can separate those explanations.
- **Decision:** State whether the next stage is approved, paused, or needs investigation.
- **Limitations:** Note the grain, denominator, join path, and whether the result is descriptive or associative.
- **📸 Screenshot:** Capture only if the result provides a clear, verified technical story worth sharing later; otherwise no screenshot is needed.

## AN1.01 — Student key completeness and uniqueness

**Question / Why:** Check the three-column key `(code_module, code_presentation, id_student)` separately in both student tables.

**Expectation:** No incomplete keys or duplicate key groups.

**Read as:** AN2 profile queries are gated on both tables returning PASS.

In [ ]:
# 1. Execute the canonical SQL file and display only its bounded result set.
result = run_query("AN1.01")
query_results["AN1.01"] = result
display(result["data"])
print(f"Query file: {result['query_file']}")
print(f"Elapsed seconds: {result['elapsed_seconds']}")
print(f"Displayed rows: {result['rows_displayed']}; truncated: {result['display_truncated']}")

**After execution — interpretation and decision (complete after reviewing the output):**

- **Observed:** Record the actual values and any difference from expectation.
- **Alternative explanations:** If results differ, state at least two plausible explanations.
- **Discriminating evidence:** Identify the next read-only check that can separate those explanations.
- **Decision:** State whether the next stage is approved, paused, or needs investigation.
- **Limitations:** Note the grain, denominator, join path, and whether the result is descriptive or associative.
- **📸 Screenshot:** Capture only if the result provides a clear, verified technical story worth sharing later; otherwise no screenshot is needed.

## AN1.02 — Student key-set reconciliation

**Question / Why:** Compare the complete composite keys in both directions, as well as distinct student IDs.

**Expectation:** No keys exclusive to either table if both represent the same student-presentation population.

**Read as:** Interpret this only after reviewing AN1.01.

In [ ]:
# 1. Execute the canonical SQL file and display only its bounded result set.
result = run_query("AN1.02")
query_results["AN1.02"] = result
display(result["data"])
print(f"Query file: {result['query_file']}")
print(f"Elapsed seconds: {result['elapsed_seconds']}")
print(f"Displayed rows: {result['rows_displayed']}; truncated: {result['display_truncated']}")

**After execution — interpretation and decision (complete after reviewing the output):**

- **Observed:** Record the actual values and any difference from expectation.
- **Alternative explanations:** If results differ, state at least two plausible explanations.
- **Discriminating evidence:** Identify the next read-only check that can separate those explanations.
- **Decision:** State whether the next stage is approved, paused, or needs investigation.
- **Limitations:** Note the grain, denominator, join path, and whether the result is descriptive or associative.
- **📸 Screenshot:** Capture only if the result provides a clear, verified technical story worth sharing later; otherwise no screenshot is needed.

## AN1.03 — Assessment join fan-out

**Question / Why:** Compare joining on `id_student` alone with the correct route through `assessments`, matching module and presentation.

**Expectation:** The student-only join may inflate rows because a student can appear in multiple presentations.

**Read as:** A multiplier above 1 indicates join fan-out, not additional assessment records. Review unmatched counts too.

In [ ]:
# 1. Execute the canonical SQL file and display only its bounded result set.
result = run_query("AN1.03")
query_results["AN1.03"] = result
display(result["data"])
print(f"Query file: {result['query_file']}")
print(f"Elapsed seconds: {result['elapsed_seconds']}")
print(f"Displayed rows: {result['rows_displayed']}; truncated: {result['display_truncated']}")

**After execution — interpretation and decision (complete after reviewing the output):**

- **Observed:** Record the actual values and any difference from expectation.
- **Alternative explanations:** If results differ, state at least two plausible explanations.
- **Discriminating evidence:** Identify the next read-only check that can separate those explanations.
- **Decision:** State whether the next stage is approved, paused, or needs investigation.
- **Limitations:** Note the grain, denominator, join path, and whether the result is descriptive or associative.
- **📸 Screenshot:** Capture only if the result provides a clear, verified technical story worth sharing later; otherwise no screenshot is needed.

## AN2.01 — Student counts by presentation

**Question / Why:** Compare student-info and registration counts at module-presentation grain.

**Expectation:** Counts should reconcile when key populations match.

**Read as:** This cell runs only if AN1.01 passes for both student tables.

In [ ]:
# 1. Run this profile only when both student tables passed AN1.01.
gate = query_results.get("AN1.01")
if gate is None or len(gate["data"]) != 2 or not (gate["data"]["gate_status"] == "PASS").all():
    print("SKIPPED: AN1.01 did not confirm complete, unique keys in both student tables.")
else:
    result = run_query("AN2.01")
    query_results["AN2.01"] = result
    display(result["data"])
    print(f"Query file: {result['query_file']}")
    print(f"Elapsed seconds: {result['elapsed_seconds']}")
    print(f"Displayed rows: {result['rows_displayed']}; truncated: {result['display_truncated']}")

**After execution — interpretation and decision (complete after reviewing the output):**

- **Observed:** Record the actual values and any difference from expectation.
- **Alternative explanations:** If results differ, state at least two plausible explanations.
- **Discriminating evidence:** Identify the next read-only check that can separate those explanations.
- **Decision:** State whether the next stage is approved, paused, or needs investigation.
- **Limitations:** Note the grain, denominator, join path, and whether the result is descriptive or associative.
- **📸 Screenshot:** Capture only if the result provides a clear, verified technical story worth sharing later; otherwise no screenshot is needed.

## AN2.02 — Final-result distribution

**Question / Why:** Describe the observed outcome categories within each presentation, including missing outcomes.

**Expectation:** Within-presentation percentages sum to approximately 100%.

**Read as:** This is descriptive, not causal. The denominator includes all rows in `student_info` for that presentation.

In [ ]:
# 1. Run this profile only when both student tables passed AN1.01.
gate = query_results.get("AN1.01")
if gate is None or len(gate["data"]) != 2 or not (gate["data"]["gate_status"] == "PASS").all():
    print("SKIPPED: AN1.01 did not confirm complete, unique keys in both student tables.")
else:
    result = run_query("AN2.02")
    query_results["AN2.02"] = result
    display(result["data"])
    print(f"Query file: {result['query_file']}")
    print(f"Elapsed seconds: {result['elapsed_seconds']}")
    print(f"Displayed rows: {result['rows_displayed']}; truncated: {result['display_truncated']}")

**After execution — interpretation and decision (complete after reviewing the output):**

- **Observed:** Record the actual values and any difference from expectation.
- **Alternative explanations:** If results differ, state at least two plausible explanations.
- **Discriminating evidence:** Identify the next read-only check that can separate those explanations.
- **Decision:** State whether the next stage is approved, paused, or needs investigation.
- **Limitations:** Note the grain, denominator, join path, and whether the result is descriptive or associative.
- **📸 Screenshot:** Capture only if the result provides a clear, verified technical story worth sharing later; otherwise no screenshot is needed.

## Work-order acceptance criteria

WO-1 remains in progress until all eligible queries have returned results, interpretations have been reviewed, decisions have been recorded in `docs/analysis_log.md`, local changes have been reviewed with `git diff`, and the progress ledger has been updated. AN2 queries may remain skipped when the key-validation gate does not pass.

Only aggregate query outputs and relevant error messages belong in the review record. Environment files, passwords, private connection strings, and raw CSV files are excluded.